---
## 3. Data Cleaning (Tickets & Surcharges Dataset)

We will now clean the primary dataset used for the main analysis.

In [6]:
import pandas as pd
import numpy as np
import os

# Load the merged dataset
df = pd.read_csv('../data/processed/3_ticket_prices_and_surcharges.csv')
print("Original Shape:", df.shape)
df.head()

Original Shape: (14355, 25)


,month,conflict_phase,airline,iata_code,country,region,airline_type,route_class,avg_route_km,base_fare_usd,...,load_factor_pct,fuel_cost_pct_opex,yoy_price_change_pct,km_range,surcharge_band,fuel_surcharge_usd_surcharge_policy,brent_crude_usd_barrel,jet_fuel_usd_barrel_surcharge_policy,surcharge_as_pct_base,yoy_surcharge_change_pct
0,2019-01,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1179.91,...,79.4,0.209,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-02,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1176.08,...,89.0,0.216,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-03,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1133.88,...,64.1,0.246,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-04,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1237.95,...,75.1,0.225,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-05,Pre-Pandemic Baseline,ANA,NH,Japan,Asia,Flag Carrier,Long-Haul,8500,1270.08,...,85.8,0.268,NaN,4501–9000 km,NaN,NaN,NaN,NaN,NaN,NaN


### 3.1 Column Name Standardisation

In [7]:
# Strip whitespace and lowercase all column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Drop fully-redundant duplicate columns that came from the merge
cols_to_drop = [
    'brent_crude_usd_barrel',              # duplicate of brent_crude_usd
    'jet_fuel_usd_barrel_surcharge_policy' # duplicate of jet_fuel_usd_barrel
]
existing_drops = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=existing_drops, inplace=True)

print(f'Columns after cleanup: {df.shape[1]}')
df.columns.tolist()

Columns after cleanup: 23


['month',
 'conflict_phase',
 'airline',
 'iata_code',
 'country',
 'region',
 'airline_type',
 'route_class',
 'avg_route_km',
 'base_fare_usd',
 'fuel_surcharge_usd',
 'taxes_fees_usd',
 'total_fare_usd',
 'brent_crude_usd',
 'jet_fuel_usd_barrel',
 'load_factor_pct',
 'fuel_cost_pct_opex',
 'yoy_price_change_pct',
 'km_range',
 'surcharge_band',
 'fuel_surcharge_usd_surcharge_policy',
 'surcharge_as_pct_base',
 'yoy_surcharge_change_pct']

### 3.2 Date / Time Parsing

In [8]:
# Parse 'month' (format: YYYY-MM) to datetime
df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

# Extract useful time columns
df['year']      = df['month'].dt.year
df['month_num'] = df['month'].dt.month
df['quarter']   = df['month'].dt.to_period('Q').astype(str)

print('Date range:', df['month'].min().strftime('%Y-%m'), '→', df['month'].max().strftime('%Y-%m'))

Date range: 2019-01 → 2026-03


### 3.3 Handle Infinite Values

In [9]:
# Replace ±Inf with NaN so they are treated as missing
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = {}
for col in numeric_cols:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        inf_counts[col] = n_inf
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

print('Columns with Inf values (now replaced with NaN):')
for k, v in inf_counts.items():
    print(f'  {k}: {v} Inf values')

Columns with Inf values (now replaced with NaN):
  yoy_surcharge_change_pct: 1250 Inf values


### 3.4 Missing Value Imputation

In [10]:
# ── yoy_price_change_pct ──────────────────────────────────────────────────────
# First 12 months per airline×route_class naturally have no YoY value.
# Forward-fill within each airline+route_class group.
df = df.sort_values(['airline', 'route_class', 'month']).reset_index(drop=True)
df['yoy_price_change_pct'] = (
    df.groupby(['airline', 'route_class'])['yoy_price_change_pct']
      .transform(lambda s: s.ffill())
      .fillna(0)
)

# ── surcharge_band ────────────────────────────────────────────────────────────
# Fill missing with 'No Surcharge' where regime doesn't apply
if 'surcharge_band' in df.columns:
    df['surcharge_band'] = df['surcharge_band'].fillna('No Surcharge')

# ── numeric surcharge policy cols ─────────────────────────────────────────────
# Fill with 0 (no surcharge applied)
surcharge_numeric = ['fuel_surcharge_usd_surcharge_policy', 'surcharge_as_pct_base']
for col in surcharge_numeric:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# ── yoy_surcharge_change_pct ──────────────────────────────────────────────────
if 'yoy_surcharge_change_pct' in df.columns:
    df['yoy_surcharge_change_pct'] = (
        df.groupby(['airline', 'route_class'])['yoy_surcharge_change_pct']
          .transform(lambda s: s.ffill())
          .fillna(0)
    )

# Final missing check
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
print('Remaining missing values after imputation:')


Remaining missing values after imputation:


### 3.5 Outlier Detection (Flag, Not Remove)

In [11]:
# IQR-based outlier report for key numeric columns
key_numeric = [
    'base_fare_usd', 'fuel_surcharge_usd', 'total_fare_usd',
    'brent_crude_usd', 'jet_fuel_usd_barrel', 'load_factor_pct'
]

outlier_summary = []
for col in key_numeric:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({
        'Column': col, 'Q1': round(Q1, 2), 'Q3': round(Q3, 2),
        'IQR': round(IQR, 2), 'Lower Fence': round(lower, 2),
        'Upper Fence': round(upper, 2), 'Outliers (IQR)': n_out
    })

out_df = pd.DataFrame(outlier_summary).set_index('Column')

# Outliers are NOT removed — they represent genuine extreme events
# (COVID-19, Ukraine War, US-Iran conflict spikes). Flag only.
df['is_extreme_fare'] = (
    (df['total_fare_usd'] > df['total_fare_usd'].quantile(0.99)) |
    (df['total_fare_usd'] < df['total_fare_usd'].quantile(0.01))
)
print(f'Extreme fare rows flagged: {df["is_extreme_fare"].sum()}')


Extreme fare rows flagged: 287


### 3.6 Data Type Finalisation

In [12]:
# Convert categoricals to efficient dtype
cat_cols = [
    'conflict_phase', 'airline', 'iata_code', 'country',
    'region', 'airline_type', 'route_class', 'km_range', 'surcharge_band'
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

print('Final dtypes setup for memory efficiency.')

Final dtypes setup for memory efficiency.


---
## 4. Feature Engineering

In [13]:
# Fuel surcharge as share of total fare
df['fuel_surcharge_ratio'] = (df['fuel_surcharge_usd'] / df['total_fare_usd']).round(4)

# Taxes share
df['taxes_ratio'] = (df['taxes_fees_usd'] / df['total_fare_usd']).round(4)

# Base fare share
df['base_ratio'] = (df['base_fare_usd'] / df['total_fare_usd']).round(4)

# Crude-to-jet ratio (refinery margin proxy)
df['crude_jet_ratio'] = (df['jet_fuel_usd_barrel'] / df['brent_crude_usd']).round(4)

# Fare per km (yield metric)
df['fare_per_km'] = (df['total_fare_usd'] / df['avg_route_km']).round(4)

new_feats = ['fuel_surcharge_ratio', 'taxes_ratio', 'base_ratio', 'crude_jet_ratio', 'fare_per_km']
print('New features added.')

New features added.


---
## 5. Export Cleaned Datasets

Exporting the merged and cleaned datasets into the `data/processed/` folder for use in modeling and dashboarding.

In [14]:
os.makedirs('../data/cleaned_dataset', exist_ok=True)

# Export full cleaned dataset (the main one)
EXPORT_PATH = '../data/cleaned_dataset/cleaned_ticket_prices.csv'
df_export = df.copy()
if 'month' in df_export.columns and pd.api.types.is_datetime64_any_dtype(df_export['month']):
    df_export['month'] = df_export['month'].dt.strftime('%Y-%m')
for col in df_export.select_dtypes('category').columns:
    df_export[col] = df_export[col].astype(str)

df_export.to_csv(EXPORT_PATH, index=False)

print("Process complete. Exported file:")


Process complete. Exported file:
- cleaned_ticket_prices.csv (Shape: (14355, 32))
